# Vérification du Geoprocessing Vectoriel Métier

Ce notebook démontre les **3 piliers** exigés :
1. **Buffers réglementaires** (oueds 50/100/200 m + emprise voirie)
2. **Overlays** (Intersection pour densite_bati/routes, Union pour pct_impermeable)
3. **Analyse Réseau** (nodage, graphe, Dijkstra multi-sources → distance_secours)



## 0. Configuration et imports

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import shapely
from scipy.spatial import cKDTree

# --- Racine du projet
ROOT = Path.cwd()
if ROOT.name == "scripts":
    ROOT = ROOT.parent

DOSSIER_CLEAN   = ROOT / "donnees_traitees" / "casablanca_settat_clean"
DOSSIER_MAILLES = ROOT / "donnees_traitees" / "mailles"

FICHIER_MAILLES   = DOSSIER_MAILLES / "mailles_100m.gpkg"
FICHIER_COURS_EAU = DOSSIER_CLEAN / "gis_osm_waterways_casablanca_settat_clean.gpkg"
FICHIER_ROUTES    = DOSSIER_CLEAN / "routes_casablanca_settat_reprojete.gpkg"
FICHIER_BATI      = DOSSIER_CLEAN / "bati_casablanca_settat_clean.gpkg"
FICHIER_OCSOL     = DOSSIER_CLEAN / "occupation_sol_casablanca_settat_clean.gpkg"
FICHIER_SECOURS   = DOSSIER_CLEAN / "infrastructures_secours_clean.gpkg"

CRS_PROJET = 26191          # EPSG:26191 – Merchich / Nord Maroc (mètres)
TAILLE_MAILLE = 100
LARGEURS_TAMPON_M = [50, 100, 200]

print("ROOT :", ROOT)
print("Configuration OK")

ROOT : C:\Users\dell\Documents\geoai-inondation-casablanca
Configuration OK


## 1. Lecture des données (avec vérification CRS systématique)

In [2]:
# --- Mailles ---
mailles = gpd.read_file(FICHIER_MAILLES, layer="maille")
if "geometry" in mailles.columns and "geom" not in mailles.columns:
    mailles = mailles.rename_geometry("geom")
elif "geom" in mailles.columns and mailles.geometry.name != "geom":
    mailles = mailles.set_geometry("geom")
assert mailles.crs.to_epsg() == CRS_PROJET, "Mailles : CRS différent de 26191"

# Centroïdes (stockés en WKT dans la table Maille)
if isinstance(mailles["centroide"].iloc[0], str):
    centroides = gpd.GeoSeries.from_wkt(mailles["centroide"], crs=CRS_PROJET)
else:
    centroides = mailles["geom"].centroid

# --- Fonction utilitaire ---
def lire_et_reprojeter(fichier, nom):
    """Lit un GeoPackage et reprojette en EPSG:26191 si nécessaire."""
    gdf = gpd.read_file(fichier)
    if gdf.crs is None or gdf.crs.to_epsg() != CRS_PROJET:
        gdf = gdf.to_crs(CRS_PROJET)
    print(f"  {nom:20s} : {len(gdf):>10,} entités | CRS EPSG:{gdf.crs.to_epsg()}")
    return gdf

cours_eau = lire_et_reprojeter(FICHIER_COURS_EAU, "Cours d'eau")
routes    = lire_et_reprojeter(FICHIER_ROUTES,    "Routes")
bati      = lire_et_reprojeter(FICHIER_BATI,      "Bâtiments")
ocsol     = lire_et_reprojeter(FICHIER_OCSOL,     "Occupation du sol")
secours   = lire_et_reprojeter(FICHIER_SECOURS,   "Secours")

print(f"  {'Mailles':20s} : {len(mailles):>10,} entités | CRS EPSG:{mailles.crs.to_epsg()}")

  Cours d'eau          :      1,449 entités | CRS EPSG:26191
  Routes               :      4,290 entités | CRS EPSG:26191
  Bâtiments            :     84,962 entités | CRS EPSG:26191
  Occupation du sol    :     11,810 entités | CRS EPSG:26191
  Secours              :         50 entités | CRS EPSG:26191
  Mailles              :  2,020,705 entités | CRS EPSG:26191


---
# PILIER 1 : Buffers Réglementaires

## 1A. Buffers autour des cours d'eau (50, 100, 200 m)

Bandes d'étude autour des oueds. Les buffers sont **dissous** (`union_all`) pour éliminer les chevauchements entre tronçons adjacents.

In [3]:
print("--- Buffers autour des cours d'eau ---\n")

bandes = {}
x = shapely.get_x(centroides.values)
y = shapely.get_y(centroides.values)

for w in LARGEURS_TAMPON_M:
    # Union de tous les buffers → 1 seul polygone sans chevauchement
    bandes[w] = shapely.union_all(
        cours_eau.geometry.buffer(w, quad_segs=32).values
    )
    # Comptage des centroïdes de mailles dans chaque bande
    dans_bande = shapely.contains_xy(bandes[w], x, y)
    print(f"  Bande {w:>3} m : {dans_bande.sum():>7,} centroïdes dans la zone tampon")

--- Buffers autour des cours d'eau ---

  Bande  50 m :  15,287 centroïdes dans la zone tampon
  Bande 100 m :  30,328 centroïdes dans la zone tampon
  Bande 200 m :  59,512 centroïdes dans la zone tampon


## 1B. Buffer d'emprise de la voirie

Chaque route est bufferisée selon le nombre de voies (`lanes × 3.5 m / 2`). Le résultat sert aussi au calcul de `pct_impermeable` (Pilier 2).

In [4]:
print("--- Buffer d'emprise de la voirie ---\n")

lanes = pd.to_numeric(routes.get("lanes", 2), errors="coerce").fillna(2.0)
lanes = np.where(lanes <= 0, 2.0, lanes)
rayons = (lanes * 3.5) / 2.0  # demi-largeur = rayon du buffer

emprise_routes = routes.geometry.buffer(rayons)
gdf_routes_poly = gpd.GeoDataFrame(
    {"geometry": emprise_routes[~emprise_routes.is_empty]}, crs=CRS_PROJET
)

print(f"  Polygones d'emprise créés : {len(gdf_routes_poly):,}")
print(f"  Largeur moyenne emprise   : {(lanes * 3.5).mean():.1f} m")

--- Buffer d'emprise de la voirie ---

  Polygones d'emprise créés : 4,290
  Largeur moyenne emprise   : 8.8 m


---
# PILIER 2 : Overlays Métier (Intersection & Union)

## 2A. INTERSECTION : densité bâtie par maille (`densite_bati`)

Chaque polygone de bâtiment est **découpé** (`gpd.overlay(..., how="intersection")`) par les mailles. La surface découpée est sommée par maille puis divisée par la surface de la maille.

In [5]:
print("--- INTERSECTION : densite_bati ---\n")

mailles_geom = mailles[["id", "geom"]].copy()
surface_maille = mailles["geom"].area  # 10 000 m² pour 100 m

# Filtre spatial (bounding box)
bbox = mailles.total_bounds
bati_filtre = bati.cx[bbox[0]:bbox[2], bbox[1]:bbox[3]].copy()

# Overlay intersection : chaque morceau de bâtiment reçoit l'id de la maille
morceaux_bati = gpd.overlay(
    bati_filtre[["geometry"]],
    mailles_geom,
    how="intersection",
    keep_geom_type=True,
)
morceaux_bati["surf_morceau_m2"] = morceaux_bati.geometry.area

# Somme des surfaces par maille → ratio
surf_par_maille = morceaux_bati.groupby("id")["surf_morceau_m2"].sum()
surface_bati_totale = mailles["id"].map(surf_par_maille).fillna(0.0)
densite_bati = (surface_bati_totale / surface_maille).clip(lower=0.0, upper=1.0)

print(f"  Mailles avec bâti : {(densite_bati > 0).sum():,} / {len(mailles):,}")
print(f"  Densité moyenne    : {densite_bati.mean():.4f}")
print(f"  Densité maximale   : {densite_bati.max():.4f}")

--- INTERSECTION : densite_bati ---

  Mailles avec bâti : 20,512 / 2,020,705
  Densité moyenne    : 0.0020
  Densité maximale   : 1.0000


## 2B. INTERSECTION : densité routière par maille (`densite_routes`)

Chaque tronçon routier est découpé par les mailles traversées. La longueur découpée est sommée par maille.

In [6]:
print("--- INTERSECTION : densite_routes ---\n")

# Pré-filtre : mailles qui intersectent au moins une route
idx_mailles_routes = gpd.sjoin(
    mailles_geom, routes[["geometry"]], how="inner", predicate="intersects"
).index.unique()
mailles_touchees = mailles_geom.loc[idx_mailles_routes]

# Overlay intersection
morceaux_routes = gpd.overlay(
    routes[["geometry"]], mailles_touchees, how="intersection", keep_geom_type=True
)
morceaux_routes["longueur_m"] = morceaux_routes.geometry.length

longueur_par_maille = morceaux_routes.groupby("id")["longueur_m"].sum()

print(f"  Mailles traversées par une route : {len(mailles_touchees):,}")
print(f"  Morceaux découpés                : {len(morceaux_routes):,}")
print(f"  Longueur totale découpée         : {longueur_par_maille.sum() / 1000:,.1f} km")

--- INTERSECTION : densite_routes ---

  Mailles traversées par une route : 8,906
  Morceaux découpés                : 20,165
  Longueur totale découpée         : 1,232.2 km


## 2C. UNION : pourcentage imperméable par maille (`pct_impermeable`)

**Union** des 3 couches imperméables (sol classé imperméable + bâti + emprise voirie), puis **intersection** avec les mailles.

L'union est faite **localement par maille** (pas globalement) pour éviter l'explosion mémoire.

In [7]:
print("--- UNION + INTERSECTION : pct_impermeable ---\n")

# 1) Couches imperméables
ocsol_valid = ocsol[ocsol.geometry.is_valid & ~ocsol.geometry.is_empty].copy()
sol_imper = ocsol_valid[ocsol_valid["landuse_class"] == "impermeable"][["geometry"]].copy()
bati_poly = bati_filtre[["geometry"]].copy()

# 2) Concaténation des 3 couches (PAS d'union globale géante !)
couches_imper = pd.concat([sol_imper, bati_poly, gdf_routes_poly], ignore_index=True)
couches_imper = gpd.GeoDataFrame(couches_imper, geometry="geometry", crs=CRS_PROJET)
print(f"  Polygones imperméables concaténés : {len(couches_imper):,}")

# 3) Filtre spatial R-Tree : seules les mailles qui touchent l'imperméable
mailles_candidates_idx = mailles_geom.sjoin(
    couches_imper, how="inner", predicate="intersects"
).index.unique()
mailles_actives = mailles_geom.loc[mailles_candidates_idx]
print(f"  Mailles concernées : {len(mailles_actives):,} / {len(mailles):,}")

# 4) Découpage par overlay intersection
print("  Découpage géométrique ciblé...")
morceaux_imper = gpd.overlay(
    couches_imper, mailles_actives, how="intersection", keep_geom_type=True
)

# 5) UNION LOCALE PAR MAILLE (élimine les chevauchements sans exploser la mémoire)
print("  Union locale par maille...")
union_par_maille = morceaux_imper.groupby("id")["geometry"].apply(shapely.union_all)
surf_imper_par_maille = union_par_maille.area

# 6) Calcul du pourcentage
surf_imper_totale = mailles["id"].map(surf_imper_par_maille).fillna(0.0)
pct_impermeable = (surf_imper_totale / surface_maille * 100.0).clip(0.0, 100.0)

print(f"\n  Imperméabilité moyenne     : {pct_impermeable.mean():.2f} %")
print(f"  Mailles > 50% imperméable : {(pct_impermeable > 50).sum():,}")

--- UNION + INTERSECTION : pct_impermeable ---

  Polygones imperméables concaténés : 93,846
  Mailles concernées : 64,332 / 2,020,705
  Découpage géométrique ciblé...
  Union locale par maille...

  Imperméabilité moyenne     : 1.66 %
  Mailles > 50% imperméable : 30,890


---
# PILIER 3 : Analyse Réseau (Accessibilité Secours)

## 3A. Nodage des intersections routières

Les routes sont fusionnées puis découpées à chaque intersection avec `shapely.node`, créant de vrais carrefours topologiques.

In [8]:
print("--- Nodage des intersections routières ---\n")

lignes_routes = routes.geometry[
    routes.geometry.is_valid & ~routes.geometry.is_empty
].values

# Fusion puis découpe stricte à chaque intersection
lignes_unies = shapely.union_all(lignes_routes)
lignes_nodees = shapely.node(lignes_unies)

# Extraction des segments simples (LineString)
if lignes_nodees.geom_type == "MultiLineString":
    segments = list(lignes_nodees.geoms)
elif lignes_nodees.geom_type == "LineString":
    segments = [lignes_nodees]
else:
    segments = [g for g in lignes_nodees.geoms if g.geom_type == "LineString"]

print(f"  Segments après nodage : {len(segments):,}")

--- Nodage des intersections routières ---

  Segments après nodage : 5,313


## 3B. Construction du graphe routier

Les coordonnées sont arrondies à 2 décimales pour fusionner les nœuds quasi-coïncidents. Seule la **composante connexe principale** est conservée (suppression des tronçons isolés).

In [9]:
print("--- Construction du graphe routier ---\n")

G = nx.Graph()
for seg in segments:
    coords = list(seg.coords)
    for i in range(len(coords) - 1):
        # Arrondi pour fusionner les nœuds quasi-coïncidents
        u = (round(coords[i][0], 2), round(coords[i][1], 2))
        v = (round(coords[i + 1][0], 2), round(coords[i + 1][1], 2))
        longueur = float(shapely.Point(u).distance(shapely.Point(v)))
        if longueur > 0:
            G.add_edge(u, v, weight=longueur)

print(f"  Graphe brut : {G.number_of_nodes():,} nœuds, {G.number_of_edges():,} arêtes")

# Composante connexe principale
composantes = list(nx.connected_components(G))
plus_grande = max(composantes, key=len)
G_principal = G.subgraph(plus_grande).copy()

print(
    f"  Composante principale : {G_principal.number_of_nodes():,} nœuds"
    f" ({len(G_principal) / len(G) * 100:.1f}% du réseau conservé)"
)

--- Construction du graphe routier ---

  Graphe brut : 35,949 nœuds, 37,150 arêtes
  Composante principale : 35,881 nœuds (99.8% du réseau conservé)


## 3C. Index spatial KD-Tree et projection des casernes

In [10]:
print("--- Index KD-Tree et projection des casernes ---\n")

noeuds_coords = np.array(list(G_principal.nodes()))
kdtree = cKDTree(noeuds_coords)
print(f"  KD-Tree construit sur {len(noeuds_coords):,} nœuds")

# Projection de chaque caserne sur le nœud routier le plus proche
coords_secours = np.array([[pt.x, pt.y] for pt in secours.geometry])
dist_sec_noeud, idx_sec_noeud = kdtree.query(coords_secours)

# Nœuds d'entrée (si plusieurs casernes sur le même nœud, garder la plus proche)
sources_secours = {}
for i, idx in enumerate(idx_sec_noeud):
    noeud = tuple(noeuds_coords[idx])
    d_approche = dist_sec_noeud[i]
    if noeud not in sources_secours or d_approche < sources_secours[noeud]:
        sources_secours[noeud] = d_approche

print(
    f"  {len(secours)} casernes rattachées à"
    f" {len(sources_secours)} nœuds d'entrée distincts"
)

--- Index KD-Tree et projection des casernes ---

  KD-Tree construit sur 35,881 nœuds
  50 casernes rattachées à 44 nœuds d'entrée distincts


## 3D. Dijkstra multi-sources (casernes → tout le réseau)

Un seul appel `nx.multi_source_dijkstra_path_length` calcule la distance minimale réseau de **chaque nœud du graphe** vers la caserne la plus proche.

In [11]:
print("--- Dijkstra multi-sources ---\n")

print("  Exécution de Dijkstra global...")
distances_reseau = nx.multi_source_dijkstra_path_length(
    G_principal, sources=sources_secours.keys(), weight="weight"
)

# Ajout de la pénalité d'accès (caserne → nœud le plus proche)
for noeud_source, d_init in sources_secours.items():
    distances_reseau[noeud_source] += d_init

print(f"  Distances calculées pour {len(distances_reseau):,} nœuds du réseau")

--- Dijkstra multi-sources ---

  Exécution de Dijkstra global...
  Distances calculées pour 35,881 nœuds du réseau


## 3E. Distance totale : centroïde → nœud routier → caserne

In [12]:
print("--- Distance totale par maille ---\n")

coords_mailles = np.array([[pt.x, pt.y] for pt in centroides])

# Nœud routier le plus proche de chaque centroïde
dist_maille_noeud, idx_maille_noeud = kdtree.query(coords_mailles, workers=-1)

# Distance réseau du nœud de rattachement vers la caserne
noeuds_mailles = [tuple(noeuds_coords[idx]) for idx in idx_maille_noeud]
dist_reseau_noeud = np.array(
    [distances_reseau.get(n, np.nan) for n in noeuds_mailles]
)

# Distance totale = centroïde→nœud + nœud→caserne (via réseau)
distance_secours = dist_maille_noeud + dist_reseau_noeud

# Fallback pour les mailles non connectées au réseau
nb_nan = np.isnan(distance_secours).sum()
if nb_nan > 0:
    print(f"  ⚠ {nb_nan} mailles non connectées : imputation par distance euclidienne")
    union_secours = shapely.union_all(secours.geometry.values)
    d_directe = centroides.distance(union_secours)
    distance_secours = np.where(
        np.isnan(distance_secours), d_directe.values, distance_secours
    )

print(f"  Distance min  : {np.nanmin(distance_secours):>10,.0f} m ({np.nanmin(distance_secours)/1000:.1f} km)")
print(f"  Distance moy  : {np.nanmean(distance_secours):>10,.0f} m ({np.nanmean(distance_secours)/1000:.1f} km)")
print(f"  Distance max  : {np.nanmax(distance_secours):>10,.0f} m ({np.nanmax(distance_secours)/1000:.1f} km)")
print(f"  Valeurs NaN   : {np.isnan(distance_secours).sum()}")

--- Distance totale par maille ---

  Distance min  :         24 m (0.0 km)
  Distance moy  :     91,195 m (91.2 km)
  Distance max  :    252,799 m (252.8 km)
  Valeurs NaN   : 0


---
# Résumé final

In [13]:
print("=" * 70)
print("RÉSUMÉ – GEOPROCESSING VECTORIEL MÉTIER")
print("=" * 70)
print()
print("1. BUFFERS RÉGLEMENTAIRES")
print(f"   • Oueds  : 3 bandes (50, 100, 200 m) avec shapely.union_all")
print(f"   • Voirie : emprise = lanes × 3.5 m (buffer/2) → {len(gdf_routes_poly):,} polygones")
print()
print("2. OVERLAYS")
print(f"   • Intersection (densite_bati)      : gpd.overlay(how='intersection')")
print(f"     → {(densite_bati > 0).sum():,} mailles avec bâti")
print(f"   • Intersection (densite_routes)     : gpd.overlay(how='intersection')")
print(f"     → {len(mailles_touchees):,} mailles traversées")
print(f"   • Union locale (pct_impermeable)    : groupby + shapely.union_all")
print(f"     → imperméabilité moyenne {pct_impermeable.mean():.2f}%")
print()
print("3. ANALYSE RÉSEAU")
print(f"   • Nodage       : shapely.node (carrefours topologiques)")
print(f"   • Graphe       : {G_principal.number_of_nodes():,} nœuds connectés")
print(f"   • KD-Tree      : scipy.spatial.cKDTree")
print(f"   • Dijkstra     : multi-sources ({len(sources_secours)} casernes)")
print(f"   • Résultat     : distance_secours pour {len(mailles):,} mailles")
print()
print("✅ Les 3 piliers du geoprocessing vectoriel métier sont vérifiés.")

RÉSUMÉ – GEOPROCESSING VECTORIEL MÉTIER

1. BUFFERS RÉGLEMENTAIRES
   • Oueds  : 3 bandes (50, 100, 200 m) avec shapely.union_all
   • Voirie : emprise = lanes × 3.5 m (buffer/2) → 4,290 polygones

2. OVERLAYS
   • Intersection (densite_bati)      : gpd.overlay(how='intersection')
     → 20,512 mailles avec bâti
   • Intersection (densite_routes)     : gpd.overlay(how='intersection')
     → 8,906 mailles traversées
   • Union locale (pct_impermeable)    : groupby + shapely.union_all
     → imperméabilité moyenne 1.66%

3. ANALYSE RÉSEAU
   • Nodage       : shapely.node (carrefours topologiques)
   • Graphe       : 35,881 nœuds connectés
   • KD-Tree      : scipy.spatial.cKDTree
   • Dijkstra     : multi-sources (44 casernes)
   • Résultat     : distance_secours pour 2,020,705 mailles

✅ Les 3 piliers du geoprocessing vectoriel métier sont vérifiés.
